In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define path to the pre-processed SMARD dataset
file_path = (
    "/content/drive/MyDrive/Colab Notebooks/SMARD_Cleaned_20260806_20260816.csv"
)

# Verify file existence
if os.path.exists(file_path):
  print("Dataset found successfully! Proceeding with data loading...")
else:
  print(
      "File not found! Please verify the folder structure in your Google Drive."
  )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset found successfully! Proceeding with data loading...


In [5]:
pip install entsoe-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 10.6 MB/s eta 0:00:00


In [8]:
import pandas as pd
from entsoe import EntsoePandasClient
import pulp

# 1. Initialize ENTSO-E Client and fetch real data with error handling
API_KEY = "45ab4959-6970-424b-9fd5-f31dda32cb2b"
client = EntsoePandasClient(api_key=API_KEY)

start = pd.Timestamp("2026-09-01 00:00:00", tz="Europe/Berlin")
end = pd.Timestamp("2026-09-01 23:00:00", tz="Europe/Berlin")
country_code = "DE_LU"

try:
    print("Fetching real-world Day-Ahead prices from ENTSO-E API...")
    prices = client.query_day_ahead_prices(country_code, start=start, end=end)
    # Ensure proper timezone localization
    prices = prices.tz_convert("Europe/Berlin")
except Exception as e:
    print(f"API connection error encountered ({e}). Using mock fallback prices for execution.")
    # Fallback synthetic prices for 24 hours if server returns 503/unavailable
    hours = pd.date_range(start=start, end=end, freq="h")
    prices = pd.Series([50, 40, 30, 25, 20, 22, 35, 60, 90, 80, 70, 60,
                        55, 50, 45, 50, 65, 100, 120, 110, 90, 75, 60, 45], index=hours)

# Extract hourly price array for optimization
price_list = prices.values[:24]
time_periods = range(len(price_list))

# 2. Define Battery Energy Storage System (BESS) Technical Parameters
max_capacity = 100.0  # Maximum energy capacity in MWh
max_power = 50.0      # Maximum charge and discharge power rate in MW
efficiency = 0.90     # Round-trip efficiency factor
eta = efficiency ** 0.5

# 3. Initialize PuLP Linear Programming Problem
prob = pulp.LpProblem("BESS_Arbitrage_Optimization", pulp.LpMaximize)

# 4. Define Decision Variables
charge = {t: pulp.LpVariable(f"charge_{t}", lowBound=0, upBound=max_power) for t in time_periods}
discharge = {t: pulp.LpVariable(f"discharge_{t}", lowBound=0, upBound=max_power) for t in time_periods}
soc = {t: pulp.LpVariable(f"soc_{t}", lowBound=0, upBound=max_capacity) for t in time_periods}

# 5. Define Objective Function: Maximize financial arbitrage revenue
prob += pulp.lpSum((discharge[t] - charge[t]) * price_list[t] for t in time_periods), "Total_Arbitrage_Profit"

# 6. Define Operational Constraints
initial_soc = 50.0  # Initial energy level in MWh at hour 0

for t in time_periods:
    if t == 0:
        prob += soc[t] == initial_soc + (charge[t] * eta) - (discharge[t] / eta), f"SoC_Balance_{t}"
    else:
        prob += soc[t] == soc[t-1] + (charge[t] * eta) - (discharge[t] / eta), f"SoC_Balance_{t}"

# 7. Solve the Optimization Problem
prob.solve(pulp.PULP_CBC_CMD(msg=False))

print(f"\nOptimization Status: {pulp.LpStatus[prob.status]}")
print(f"Total Estimated Revenue: EUR {pulp.value(prob.objective):.2f}\n")

# 8. Format and Display Hourly Dispatch Schedule Results
schedule_data = []
for t in time_periods:
    schedule_data.append({
        "Hour": t,
        "Price (EUR/MWh)": round(price_list[t], 2),
        "Charge (MW)": round(charge[t].varValue, 2),
        "Discharge (MW)": round(discharge[t].varValue, 2),
        "SoC (MWh)": round(soc[t].varValue, 2)
    })

schedule_df = pd.DataFrame(schedule_data)
print(schedule_df.to_string(index=False))

Fetching real-world Day-Ahead prices from ENTSO-E API...
API connection error encountered (503 Server Error: Service Unavailable for url: https://web-api.tp.entsoe.eu/api?documentType=A44&in_Domain=10Y1001A1001A82H&out_Domain=10Y1001A1001A82H&offset=0&contract_MarketAgreement.type=A01&classificationSequence_AttributeInstanceComponent.position=1&securityToken=45ab4959-6970-424b-9fd5-f31dda32cb2b&periodStart=202608302200&periodEnd=202609022100). Using mock fallback prices for execution.

Optimization Status: Optimal
Total Estimated Revenue: EUR 14141.00

 Hour  Price (EUR/MWh)  Charge (MW)  Discharge (MW)  SoC (MWh)
    0               50         0.00           47.43       0.00
    1               40         0.00            0.00       0.00
    2               30         0.00            0.00       0.00
    3               25         5.41            0.00       5.13
    4               20        50.00            0.00      52.57
    5               22        50.00            0.00     100.00


In [9]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import pandas as pd

# 1. Set dark theme matching Streamlit dashboard background (#0e1117)
plt.style.use('dark_background')
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
fig.patch.set_facecolor('#0e1117')

for ax in [ax1, ax2, ax3]:
    ax.set_facecolor('#0e1117')
    ax.grid(True, color='#262730', linestyle='--', alpha=0.7)
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')

# Initialize empty plot lines for animation
line_price, = ax1.plot([], [], color='#00ffcc', lw=2.5, label='Price (EUR/MWh)')
line_charge, = ax2.plot([], [], color='#ff4b4b', lw=2.5, label='Charge Power (MW)')
line_discharge, = ax2.plot([], [], color='#28a745', lw=2.5, label='Discharge Power (MW)')
line_soc, = ax3.plot([], [], color='#ffaa00', lw=2.5, label='Battery SoC (MWh)')

# Configure axes labels and legends
ax1.set_ylabel('Market Price')
ax1.legend(loc='upper right', facecolor='#262730', edgecolor='none')
ax2.set_ylabel('Power (MW)')
ax2.legend(loc='upper right', facecolor='#262730', edgecolor='none')
ax3.set_ylabel('SoC (MWh)')
ax3.set_xlabel('Hour of Day')
ax3.legend(loc='upper right', facecolor='#262730', edgecolor='none')

ax1.set_xlim(0, 23)
ax1.set_ylim(schedule_df['Price (EUR/MWh)'].min() - 10, schedule_df['Price (EUR/MWh)'].max() + 10)

ax2.set_xlim(0, 23)
ax2.set_ylim(0, max_power + 5)

ax3.set_xlim(0, 23)
ax3.set_ylim(0, max_capacity + 10)

def init():
    line_price.set_data([], [])
    line_charge.set_data([], [])
    line_discharge.set_data([], [])
    line_soc.set_data([], [])
    return line_price, line_charge, line_discharge, line_soc

def animate(i):
    x_data = schedule_df['Hour'][:i+1]
    line_price.set_data(x_data, schedule_df['Price (EUR/MWh)'][:i+1])
    line_charge.set_data(x_data, schedule_df['Charge (MW)'][:i+1])
    line_discharge.set_data(x_data, schedule_df['Discharge (MW)'][:i+1])
    line_soc.set_data(x_data, schedule_df['SoC (MWh)'][:i+1])
    return line_price, line_charge, line_discharge, line_soc

# 2. Generate and save the GIF
anim = animation.FuncAnimation(fig, animate, init_func=init, frames=len(schedule_df), interval=300, blit=True)
gif_path = 'bess_optimization_animation.gif'
anim.save(gif_path, writer='pillow', fps=4)
plt.close()

print(f"Dark-themed animated GIF successfully generated and saved as '{gif_path}'!")

Dark-themed animated GIF successfully generated and saved as 'bess_optimization_animation.gif'!
